# Context-Aware ICD-10 Coding from Clinical Notes
## Approach Evaluation: Rule-Based vs LLM (Diabetes Focus)

**Objective:** Extract all relevant ICD-10 codes from clinical notes with context awareness  
**Scope:** Diabetes (E08-E13) + secondary/supporting codes (R, Z, M codes)  
**Approaches:** A (Rule-based baseline) vs C (LLM structured extraction)  
**Gold Standard:** Human-annotated clinical notes with evidence spans

---

## 1. Setup & Configuration

In [ ]:
import json 

In [46]:
import json
import re
import os
from collections import defaultdict
from typing import Optional

## 2. Gold Standard Test Data

In [47]:
# # ============================================================
# # NOTE 1: Type 1 DM, multi-complication (completed: true)
# # ============================================================
# NOTE_1 = {
#     "id": "note_01",
#     "text": """CHIEF COMPLAINT:  Followup diabetes mellitus, type 1. SUBJECTIVE:  Patient is a 34-year-old male with significant diabetic neuropathy.  He has been off on insurance for over a year.  Has been using NPH and Regular insulin to maintain his blood sugars.  States that he is deathly afraid of having a low blood sugar due to motor vehicle accident he was in several years ago.  Reports that his blood sugar dropped too low which caused the accident.  Since this point in time, he has been unwilling to let his blood sugars fall within a normal range, for fear of hypoglycemia. Also reports that he regulates his blood sugars with how he feels, rarely checking his blood sugar with a glucometer. Reports that he has been worked up extensively at hospital and was seeing an Endocrinologist at one time.  Reports that he had some indications of kidney damage when first diagnosed.  His urine microalbumin today is 100.  His last hemoglobin A1C drawn at the end of December is 11.9.  Reports that at one point, he was on Lantus which worked well and he did not worry about his blood sugars dropping too low. While using Lantus, he was able to get his hemoglobin A1C down to 7.  His last CMP shows an elevated alkaline phosphatase level of 168. He denies alcohol or drug use and is a non smoker.  Reports he quit drinking 3 years ago. I have discussed with patient that it would be appropriate to do an SGGT and hepatic panel today.  Patient also has a history of gastroparesis and impotence.  Patient requests Nexium and Viagra, neither of which are covered under the Health Plan.   Patient reports that he was in a scooter accident one week ago, fell off his scooter, hit his head.  Was not wearing a helmet.  Reports that he did not go to the emergency room and had a headache for several days after this incident.  Reports that an ambulance arrived at the scene and he was told he had a scalp laceration and to go into the emergency room.  Patient did not comply.  Reports that the headache has resolved.  Denies any dizziness, nausea, vomiting, or other neurological abnormalities. PHYSICAL EXAMINATION:  WD, WN.  Slender, 34-year-old white male.  VITAL SIGNS:  Blood sugar 145, blood pressure 120/88, heart rate 104, respirations 16.  Microalbumin 100. SKIN:  There appears to be 2 skin lacerations on the left parietal region of the scalp, each approximately 1 inch long.  No signs of infection.  Wound is closed with new granulation tissue.  Appears to be healing well.  HEENT:  Normocephalic.  PERRLA.  EOMI.  TMs pearly gray with landmarks present.  Nares patent.  Throat with no redness or swelling.  Nontender sinuses.  NECK:  Supple.  Full ROM.  No LAD. CARDIAC:   RRR.  No murmurs, rubs, or gallops.  RESPIRATORY:  CTA.  ABDOMEN:  Soft, nontender.  No HSM and no masses.  NEURO:  Significant for lower extremity numbness throughout.  Microfilament test shows more than 3 regions without sensation bilaterally.  Bottoms of feet appear calloused and dry.  Skin is intact.  There is also a small contusion on right shin which appears to be healing, less than 1/2 inch in length and 1 cm in diameter.  No signs of infection at this time and appears to be healing.  Cranial nerves 2-12 grossly nonfocal.  Cerebellar function intact demonstrated through RAM.  ASSESSMENT:  1.  Diabetes mellitus, type 1, poorly controlled.2.  Significant diabetic neuropathy with positive microalbuminuria.3.  Scalp laceration, secondary to motor vehicle accident, symptoms resolving.4.  Elevated Alk Phos, etiology unclear. PLAN:  1.  Diabetes mellitus type 1: We will follow up the elevated alkaline phosphatase with an SGGT and a hepatic function panel.  The positive microalbumin is 100 today.  He will be placed on a low dose Ace Inhibitor. I will put in a Prior Authorization for Lantus.  I have also asked the patient to keep a log of his blood sugars for 2 weeks.  Patient agrees to this.  We may need to put in a referral to Endocrinology to get him stabilized. Prescription given for Prilosec OTC for GERD symptoms. 2.  Followup scooter accident.  Lacerations on scalp and shin appear to be healing.  Discussed with patient if there are any signs of heat, swelling, infection to return to clinic. It is extremely important for him to watch these areas as he does not have feeling in the majority of his lower body."""
# }

# NOTE_1_GOLD = {
#     "note_id": "note_01",
#     "codes": [
#         {
#             "code": "E10.65",
#             "display": "Type 1 diabetes mellitus with hyperglycemia",
#             "evidence": ["diabetes mellitus", "type 1", "poorly controlled", "HbA1c 11.9"],
#             "reasoning": "Explicit T1DM + poorly controlled + HbA1c 11.9 = hyperglycemia"
#         },
#         {
#             "code": "E10.40",
#             "display": "Type 1 diabetes mellitus with diabetic neuropathy, unspecified",
#             "evidence": ["diabetic neuropathy", "lower extremity numbness", "microfilament test"],
#             "reasoning": "Explicit diabetic neuropathy in chief complaint and assessment"
#         },
#         {
#             "code": "E10.29",
#             "display": "Type 1 diabetes mellitus with other diabetic kidney complication",
#             "evidence": ["positive microalbuminuria", "kidney damage", "microalbumin 100"],
#             "reasoning": "Microalbuminuria linked to diabetes as renal complication"
#         },
#         {
#             "code": "R80.8",
#             "display": "Other proteinuria",
#             "evidence": ["microalbumin 100", "positive microalbuminuria"],
#             "reasoning": "Standalone code for the proteinuria finding"
#         },
#         {
#             "code": "Z86.39",
#             "display": "Personal history of other endocrine, nutritional and metabolic disease",
#             "evidence": ["history of gastroparesis"],
#             "reasoning": "Gastroparesis noted as historical condition linked to DM"
#         }
#     ]
# }


# # ============================================================
# # NOTE 2: Type 2 DM (inferred), hypoglycemia (completed: true)
# # ============================================================
# NOTE_2 = {
#     "id": "note_02",
#     "text": """Soap Note Note for Day of Date of Exam: [DATE_1] Subjective Data Subjective Data: Patient admitted with osteomyelitis of the left great Toe with poorly controlled diabetes. She was admitted by me and had an amputation of the left great toe yesterday. She continues on IV antibiotics. Podiatry Services planning delayed closure of the wound of the left foot tomorrow . Hopefully she can be discharged shortly thereafter. Objective Data Temperature: 98 F Pulse Rate: 91 Respiratory Rate: 18 Blood Pressure: 160/76 O2 Sat by Pulse Oximetry: 98 Objective Data: No complaints. Dressing intact left foot Assessment Assessment: Podiatry Service planning delayed closure of left foot tomorrow Plan Plan: plan to discharge tomorrow after surgical closure of the left foot wound. She will follow up with her primary care provider to better assess and treat her diabetes. That is already been planned and she has already contacted her PCP"""
# }
# NOTE_2_GOLD = {
#   "note_id": "note_02",
#   "codes": [
#     {
#       "code": "E11.65",
#       "display": "Type 2 diabetes mellitus with hyperglycemia",
#       "evidence": [
#         "poorly controlled",
#         "diabetes"
#       ],
#       "reasoning": ""
#     },
#     {
#       "code": "E11.69",
#       "display": "Type 2 diabetes mellitus with other specified complication",
#       "evidence": [
#         "osteomyelitis of the left great Toe",
#         "diabetes"
#       ],
#       "reasoning": ""
#     },
#     {
#       "code": "M86.9",
#       "display": "Osteomyelitis, unspecified",
#       "evidence": [
#         "osteomyelitis of the left great Toe"
#       ],
#       "reasoning": ""
#     }
#   ]
# }
# # ============================================================
# # NOTE 4: Type 2 DM (inferred), hypoglycemia (completed: true)
# # ============================================================
# NOTE_4 = {
#     "id": "note_04",
#     "text": """SUBJECTIVE:  I am asked to see the patient today with ongoing issues around her diabetic control.  We have been fairly aggressively, downwardly adjusting her  insulins , both the Lantus insulin, which we had been giving at night as well as her sliding scale Humalog insulin prior to meals.  Despite frequent decreases in her insulin regimen, she continues to have somewhat low blood glucoses, most notably in the morning when the glucoses have been in the 70s despite decreasing her Lantus insulin from around 84 units down to 60 units, which is a considerable change.  What I cannot explain is why her glucoses have not really climbed at all despite the decrease in insulin.  The staff reports to me that her appetite is good and that she is eating as well as ever.  I talked to Anna today.  She feels a little fatigued.  Otherwise, she is doing well. PHYSICAL EXAMINATION:  Vitals as in the chart.  The patient is a pleasant and cooperative.  She is in no apparent distress. ASSESSMENT AND PLAN:  Diabetes, still with some problematic low blood glucoses, most notably in the morning.  To address this situation, I am going to hold her Lantus insulin tonight and decrease and then change the administration time to in the morning.  She will get 55 units in the morning.  I am also decreasing once again her  Humalog sliding scale insulin prior to meals.  I will review the blood glucoses again next week."""
# }

# NOTE_4_GOLD = {
#     "note_id": "note_04",
#     "codes": [
#         {
#             "code": "E11.649",
#             "display": "Type 2 diabetes mellitus with hypoglycemia without coma",
#             "evidence": ["Diabetes", "low blood glucoses", "glucoses have been in the 70s"],
#             "reasoning": "DM type unspecified in adult -> T2DM default. Low glucoses = hypoglycemia."
#         },
#         {
#             "code": "Z79.4",
#             "display": "Long term (current) use of insulin",
#             "evidence": ["Lantus insulin", "Humalog insulin", "insulin regimen"],
#             "reasoning": "Multiple insulin medications actively being used and adjusted"
#         }
#     ]
# }

# # ============================================================
# # NOTE 13:
# # ============================================================
# NOTE_13 = {
#     "id": "note_13",
#     "text": """# Chief_Complaint\n- Follow-up for diabetes and hypertension management\n# HPI\nThe patient presents for a follow-up visit regarding their diabetes and hypertension management.\nThey report that their blood sugar has improved since starting Farxiga, which was recently approved by their insurance with a $15 copay.\nHowever, the patient states that their blood sugar is \"not perfect\" and requires careful monitoring.\nThey also mention difficulty controlling their anger when blood sugar levels fluctuate.\nRegarding hypertension, the patient reports taking glimepiride twice daily and metformin twice daily.\nThey are also taking benazepril 40 mg twice daily for blood pressure management.\nThe patient notes that their evening blood sugar levels are typically low, around 70 mg/dL, while morning levels can be as high as 150 mg/dL.\nThey take their evening medication around 5:00 PM, sometimes later if they forget.\nThe patient reports recent improvements in mobility but still experiences shortness of breath with exertion.\nThey mention that their back \"locked up\" for three days after doing some work yesterday.\nThe patient confirms they receive an annual flu shot in the fall and are due for a pneumonia booster at age 65.\nThe patient uses a CPAP machine for sleep apnea and reports purchasing a new mask recently.\nThey also use albuterol and Symbicort inhalers, primarily when going outside.\nThe patient is on lasix for fluid management but is currently not taking it due to perceived overlap with Farxiga's diuretic effects.\nThe patient mentions they are on their \"last refill\" from their gastroenterologist and plan to schedule a follow-up appointment soon.\nThey have been using fiber and stool softeners as recommended by their gastroenterologist.\n# ROS\n## General\nReports polyuria.\n## Musculoskeletal\nReports back spasms/contracture.\n## Psychiatric\nReports anger issues.\n# Past_History_Medical\n- Type 2 diabetes mellitus, managed with Farxiga, glimepiride, and metformin.\n- Hypertension, managed with benazepril and metoprolol.\n- Hyperlipidemia, managed with lovastatin.\n- Chronic back pain.\n- Obstructive sleep apnea, managed with CPAP.\n- Chronic respiratory condition, managed with albuterol nebulizer and Symbicort inhaler.\n- History of gastrointestinal issues, managed with fiber and stool softeners.\n# Past_History_Surgical\n# Past_History_Social\nPatient drives frequently for long distances.\nHas a daughter who provides support.\n# Past_History_Family\n# Allergies\n# Medications\n- Farxiga\n- Glimepiride, twice daily\n- Metformin, twice daily\n- Benazepril 40 mg, twice daily\n- Metoprolol\n- Lasix\n- Lovastatin\n- Verapamil\n- Albuterol nebulizer\n- Symbicort inhaler\n- Fiber supplement\n- Stool softener\n# Diagnostic_Testing\n- Hemoglobin A1c level is 7.6%, which, while an improvement, is still not at goal.\n- Lipid panel results are within normal limits.\n- Renal function is within normal limits.\n# Physical_Exam\n## Cardiovascular\nHeart sounds auscultated.\n## Respiratory\nLung sounds auscultated.\n# Assessment_Plan\n## Diabetes mellitus, uncontrolled\n- Continue Farxiga, glimepiride, and metformin.\n- Adjusted the timing of glimepiride and metformin administration to later in the evening or at bedtime to help mitigate the morning hyperglycemia and evening hypoglycemia.\n- Instructed patient to monitor blood glucose levels more closely, especially in the morning and evening, to identify patterns and help with dose timing adjustments.\n- Encouraged patient to continue monitoring blood pressure, as it is a lifelong condition that requires vigilance.\n- Refilled prescription for 90 days (where allowed) for most medications to align with patient's preference for fewer pharmacy visits.\n- Deferred lipid panel at next visit due to previous normal results.\n- Scheduled follow-up visit in 3 months to assess hemoglobin A1c, renal function, and medication efficacy.\n## Hypertension\n- Continue benazepril and metoprolol.\n- Increased the dosage of metoprolol to twice daily to optimize blood pressure control.\n- Refilled prescription for 90 days (where allowed) for most medications to align with patient's preference for fewer pharmacy visits.\n- Deferred lipid panel at next visit due to previous normal results.\n- Scheduled follow-up visit in 3 months to assess hemoglobin A1c, renal function, and medication efficacy.\n## Hyperlipidemia\n- Continue lovastatin.\n- Deferred lipid panel at next visit due to previous normal results.\n- Scheduled follow-up visit in 3 months to assess hemoglobin A1c, renal function, and medication efficacy.\n## Other\n- Administered influenza vaccination today in clinic.\n- Patient is due for pneumococcal vaccine booster at age 65."""
# }

# NOTE_13_GOLD = {'note_id': 'note_13_Amazon_108411',
#  'codes': [{'code': 'Z79.84',
#    'display': 'Long term (current) use of oral hypoglycemic drugs',
#    'evidence': ['Farxiga',
#     'glimepiride',
#     'Glimepiride',
#     'metformin',
#     'Metformin'],
#    'reasoning': ''},
#   {'code': 'E11.65',
#    'display': 'Type 2 diabetes mellitus with hyperglycemia',
#    'evidence': ['diabetes',
#     'diabetes mellitus',
#     'Diabetes mellitus',
#     'Type 2',
#     'hyperglycemia'],
#    'reasoning': ''},
#   {'code': 'E11.649',
#    'display': 'Type 2 diabetes mellitus with hypoglycemia without coma',
#    'evidence': ['diabetes',
#     'diabetes mellitus',
#     'Diabetes mellitus',
#     'Type 2',
#     'hypoglycemia'],
#    'reasoning': ''}]
# }

# # Combine
# TEST_SET = [
#     {"note": NOTE_1, "gold": NOTE_1_GOLD},
#     {"note": NOTE_2, "gold": NOTE_2_GOLD},
#     {"note": NOTE_4, "gold": NOTE_4_GOLD},
#     {"note": NOTE_13, "gold": NOTE_13_GOLD}
# ]

# print(f"Loaded {len(TEST_SET)} gold-standard test notes")
# for t in TEST_SET:
#     codes = [c["code"] for c in t["gold"]["codes"]]
#     print(f"  {t['note']['id']}: {len(t['gold']['codes'])} gold codes -> {codes}")

In [48]:
# #!/usr/bin/env python3
# """Generate combined TEST_SET from all annotated note pairs in a directory.

# Usage:
#     python build_test_set.py /path/to/notes/dir [output.json]

# Examples:
#     python build_test_set.py ./data
#     python build_test_set.py ./data test_set.json
#     python build_test_set.py ./data /output/path/test_set.json

# Structure expected:
#     dir/
#     ├── note_1.json
#     ├── note_1_ann.json
#     ├── note_2.json
#     ├── note_2_ann.json
#     └── ...
# """

# # import json
# # import sys
# # import os
# import glob
# # from typing import Optional


# def normalize_code(raw: str) -> str:
#     """E1065 -> E10.65, E11649 -> E11.649, R808 -> R80.8"""
#     if "." in raw:
#         return raw
#     if len(raw) >= 4:
#         return f"{raw[:3]}.{raw[3:]}"
#     return raw


# def load_pair(note_path: str, ann_path: str) -> dict:
#     """Load a note + annotation pair into TEST_SET entry format."""
#     with open(note_path) as f:
#         note = json.load(f)
#     with open(ann_path) as f:
#         ann = json.load(f)

#     completed = ann.get("completed", False)
#     doc_id = ann.get("doc_id", note.get("id", "unknown"))

#     if not completed:
#         print(f"  WARNING: {doc_id} not marked completed (including anyway)")

#     # Build span lookup
#     span_map = {s["id"]: s for s in ann.get("spans", [])}

#     # Convert annotations to gold codes
#     codes = []
#     for da in ann.get("document_annotations", []):
#         concept = da.get("concept", {})
#         code = normalize_code(concept.get("code", ""))

#         # Deduplicated evidence from spans
#         evidence = []
#         seen = set()
#         for sid in da.get("evidence_span_ids", []):
#             if sid in span_map:
#                 text = span_map[sid]["text"]
#                 if text not in seen:
#                     evidence.append(text)
#                     seen.add(text)

#         codes.append({
#             "code": code,
#             "display": concept.get("display", ""),
#             "evidence": evidence,
#             "reasoning": da.get("note", "")
#         })

#     return {
#         "note": note,
#         "gold": {
#             "note_id": doc_id,
#             "codes": codes,
#             "completed": completed
#         }
#     }


# def build_test_set(directory: str, output_path: Optional[str] = None) -> list[dict]:
#     """Scan directory for note/annotation pairs and build TEST_SET.

#     Args:
#         directory: Path to directory containing note_*.json and note_*_ann.json
#         output_path: Optional path to save TEST_SET as JSON

#     Returns:
#         TEST_SET list
#     """
#     directory = os.path.abspath(directory)

#     # Find all annotation files
#     ann_files = sorted(glob.glob(os.path.join(directory, "*_ann.json")))

#     if not ann_files:
#         print(f"No *_ann.json files found in {directory}")
#         return []

#     print(f"Found {len(ann_files)} annotation files in {directory}\n")

#     TEST_SET = []
#     skipped = []

#     for ann_path in ann_files:
#         # Derive note path: note_1_ann.json -> note_1.json
#         note_path = ann_path.replace("_ann.json", ".json")

#         if not os.path.exists(note_path):
#             print(f"  SKIPPED: {os.path.basename(ann_path)} — no matching note file")
#             skipped.append(ann_path)
#             continue

#         entry = load_pair(note_path, ann_path)
#         TEST_SET.append(entry)

#         nid = entry["gold"]["note_id"]
#         n_codes = len(entry["gold"]["codes"])
#         status = "completed" if entry["gold"]["completed"] else "incomplete"
#         codes = [c["code"] for c in entry["gold"]["codes"]]
#         print(f"  {nid} [{status}]: {n_codes} codes -> {codes}")

#     # Summary
#     print(f"\n{'='*50}")
#     print(f"TEST_SET: {len(TEST_SET)} notes loaded, {len(skipped)} skipped")
#     completed_count = sum(1 for t in TEST_SET if t["gold"]["completed"])
#     print(f"  Completed: {completed_count} | Incomplete: {len(TEST_SET) - completed_count}")

#     total_codes = sum(len(t["gold"]["codes"]) for t in TEST_SET)
#     print(f"  Total gold codes: {total_codes}")

#     # Save if output path provided
#     if output_path:
#         with open(output_path, "w") as f:
#             json.dump(TEST_SET, f, indent=2)
#         print(f"\nSaved to: {output_path}")

#     return TEST_SET


# # if __name__ == "__main__":
# #     if len(sys.argv) < 2:
# #         print("Usage: python build_test_set.py <directory> [output.json]")
# #         print("Example: python build_test_set.py ./data test_set.json")
# #         sys.exit(1)

# #     dir_path = sys.argv[1]
# #     out_path = sys.argv[2] if len(sys.argv) > 2 else None
# TEST_SET = build_test_set(r"D:\Projects\ClinicalNLP\test_docs")
# # TEST_SET

In [52]:
def build_test_set(notes_dir: str, ann_dir: str, output_path: Optional[str] = None) -> list[dict]:
    """Scan separate notes and annotation directories, match by filename, build TEST_SET.

    Args:
        notes_dir: Path to directory containing note_*.json
        ann_dir: Path to directory containing note_*_ann.json
        output_path: Optional path to save TEST_SET as JSON

    Returns:
        TEST_SET list
    """
    notes_dir = os.path.abspath(notes_dir)
    ann_dir = os.path.abspath(ann_dir)

    ann_files = sorted(glob.glob(os.path.join(ann_dir, "*_ann.json")))

    if not ann_files:
        print(f"No *_ann.json files found in {ann_dir}")
        return []

    print(f"Found {len(ann_files)} annotation files in {ann_dir}\n")

    TEST_SET = []
    skipped = []

    for ann_path in ann_files:
        # Derive note filename: note_1_ann.json -> note_1.json
        ann_filename = os.path.basename(ann_path)
        note_filename = ann_filename.replace("_ann.json", ".json")
        note_path = os.path.join(notes_dir, note_filename)

        if not os.path.exists(note_path):
            print(f"  SKIPPED: {ann_filename} — no matching {note_filename} in {notes_dir}")
            skipped.append(ann_path)
            continue

        entry = load_pair(note_path, ann_path)
        TEST_SET.append(entry)

        nid = entry["gold"]["note_id"]
        n_codes = len(entry["gold"]["codes"])
        status = "completed" if entry["gold"]["completed"] else "incomplete"
        codes = [c["code"] for c in entry["gold"]["codes"]]
        print(f"  {nid} [{status}]: {n_codes} codes -> {codes}")

    print(f"\n{'='*50}")
    print(f"TEST_SET: {len(TEST_SET)} notes loaded, {len(skipped)} skipped")
    completed_count = sum(1 for t in TEST_SET if t["gold"]["completed"])
    print(f"  Completed: {completed_count} | Incomplete: {len(TEST_SET) - completed_count}")
    total_codes = sum(len(t["gold"]["codes"]) for t in TEST_SET)
    print(f"  Total gold codes: {total_codes}")

    if output_path:
        with open(output_path, "w") as f:
            json.dump(TEST_SET, f, indent=2)
        print(f"\nSaved to: {output_path}")

    return TEST_SET

TEST_SET = build_test_set(r"D:\Projects\ClinicalNLP\test_docs\notes", r"D:\Projects\ClinicalNLP\test_docs\ann")
# TEST_SET

Found 4 annotation files in D:\Projects\ClinicalNLP\test_docs\ann

  note_13_Amazon_108411 [completed]: 3 codes -> ['Z79.84', 'E11.65', 'E11.649']
  note_01 [completed]: 5 codes -> ['E10.65', 'E10.29', 'R80.8', 'Z86.39', 'E10.40']
  note_02 [incomplete]: 3 codes -> ['E11.65', 'E11.69', 'M86.9']
  note_04 [completed]: 2 codes -> ['E11.649', 'Z79.4']

TEST_SET: 4 notes loaded, 0 skipped
  Completed: 3 | Incomplete: 1
  Total gold codes: 13


## 3. ICD-10 Reference (E08-E13 + Supporting Codes)

In [53]:
ICD10_REFERENCE = {
    # --- E10: TYPE 1 ---
    "E10.9": "Type 1 DM without complications",
    "E10.10": "Type 1 DM with ketoacidosis without coma",
    "E10.11": "Type 1 DM with ketoacidosis with coma",
    "E10.21": "Type 1 DM with diabetic nephropathy",
    "E10.22": "Type 1 DM with diabetic chronic kidney disease",
    "E10.29": "Type 1 DM with other diabetic kidney complication",
    "E10.311": "Type 1 DM with unspecified diabetic retinopathy with macular edema",
    "E10.319": "Type 1 DM with unspecified diabetic retinopathy without macular edema",
    "E10.321": "Type 1 DM with mild nonproliferative retinopathy with macular edema",
    "E10.329": "Type 1 DM with mild nonproliferative retinopathy without macular edema",
    "E10.331": "Type 1 DM with moderate nonproliferative retinopathy with macular edema",
    "E10.339": "Type 1 DM with moderate nonproliferative retinopathy without macular edema",
    "E10.341": "Type 1 DM with severe nonproliferative retinopathy with macular edema",
    "E10.349": "Type 1 DM with severe nonproliferative retinopathy without macular edema",
    "E10.351": "Type 1 DM with proliferative diabetic retinopathy with macular edema",
    "E10.359": "Type 1 DM with proliferative diabetic retinopathy without macular edema",
    "E10.36": "Type 1 DM with diabetic cataract",
    "E10.39": "Type 1 DM with other diabetic ophthalmic complication",
    "E10.40": "Type 1 DM with diabetic neuropathy, unspecified",
    "E10.41": "Type 1 DM with diabetic mononeuropathy",
    "E10.42": "Type 1 DM with diabetic polyneuropathy",
    "E10.43": "Type 1 DM with diabetic autonomic (poly)neuropathy",
    "E10.44": "Type 1 DM with diabetic amyotrophy",
    "E10.49": "Type 1 DM with other diabetic neurological complication",
    "E10.51": "Type 1 DM with diabetic peripheral angiopathy without gangrene",
    "E10.52": "Type 1 DM with diabetic peripheral angiopathy with gangrene",
    "E10.59": "Type 1 DM with other circulatory complications",
    "E10.610": "Type 1 DM with diabetic neuropathic arthropathy",
    "E10.618": "Type 1 DM with other diabetic arthropathy",
    "E10.620": "Type 1 DM with diabetic dermatitis",
    "E10.621": "Type 1 DM with foot ulcer",
    "E10.622": "Type 1 DM with other skin ulcer",
    "E10.628": "Type 1 DM with other skin complications",
    "E10.630": "Type 1 DM with periodontal disease",
    "E10.638": "Type 1 DM with other oral complications",
    "E10.641": "Type 1 DM with hypoglycemia with coma",
    "E10.649": "Type 1 DM with hypoglycemia without coma",
    "E10.65": "Type 1 DM with hyperglycemia",
    "E10.69": "Type 1 DM with other specified complication",
    "E10.8": "Type 1 DM with unspecified complications",
    # --- E11: TYPE 2 ---
    "E11.9": "Type 2 DM without complications",
    "E11.00": "Type 2 DM with hyperosmolarity without NKHHC",
    "E11.01": "Type 2 DM with hyperosmolarity with coma",
    "E11.21": "Type 2 DM with diabetic nephropathy",
    "E11.22": "Type 2 DM with diabetic chronic kidney disease",
    "E11.29": "Type 2 DM with other diabetic kidney complication",
    "E11.311": "Type 2 DM with unspecified diabetic retinopathy with macular edema",
    "E11.319": "Type 2 DM with unspecified diabetic retinopathy without macular edema",
    "E11.321": "Type 2 DM with mild nonproliferative retinopathy with macular edema",
    "E11.329": "Type 2 DM with mild nonproliferative retinopathy without macular edema",
    "E11.331": "Type 2 DM with moderate nonproliferative retinopathy with macular edema",
    "E11.339": "Type 2 DM with moderate nonproliferative retinopathy without macular edema",
    "E11.341": "Type 2 DM with severe nonproliferative retinopathy with macular edema",
    "E11.349": "Type 2 DM with severe nonproliferative retinopathy without macular edema",
    "E11.351": "Type 2 DM with proliferative diabetic retinopathy with macular edema",
    "E11.359": "Type 2 DM with proliferative diabetic retinopathy without macular edema",
    "E11.36": "Type 2 DM with diabetic cataract",
    "E11.39": "Type 2 DM with other diabetic ophthalmic complication",
    "E11.40": "Type 2 DM with diabetic neuropathy, unspecified",
    "E11.41": "Type 2 DM with diabetic mononeuropathy",
    "E11.42": "Type 2 DM with diabetic polyneuropathy",
    "E11.43": "Type 2 DM with diabetic autonomic (poly)neuropathy",
    "E11.44": "Type 2 DM with diabetic amyotrophy",
    "E11.49": "Type 2 DM with other diabetic neurological complication",
    "E11.51": "Type 2 DM with diabetic peripheral angiopathy without gangrene",
    "E11.52": "Type 2 DM with diabetic peripheral angiopathy with gangrene",
    "E11.59": "Type 2 DM with other circulatory complications",
    "E11.610": "Type 2 DM with diabetic neuropathic arthropathy",
    "E11.618": "Type 2 DM with other diabetic arthropathy",
    "E11.620": "Type 2 DM with diabetic dermatitis",
    "E11.621": "Type 2 DM with foot ulcer",
    "E11.622": "Type 2 DM with other skin ulcer",
    "E11.628": "Type 2 DM with other skin complications",
    "E11.630": "Type 2 DM with periodontal disease",
    "E11.638": "Type 2 DM with other oral complications",
    "E11.641": "Type 2 DM with hypoglycemia with coma",
    "E11.649": "Type 2 DM with hypoglycemia without coma",
    "E11.65": "Type 2 DM with hyperglycemia",
    "E11.69": "Type 2 DM with other specified complication",
    "E11.8": "Type 2 DM with unspecified complications",
    # --- E08/E09/E13 (abbreviated) ---
    "E08.9": "DM due to underlying condition without complications",
    "E08.65": "DM due to underlying condition with hyperglycemia",
    "E09.9": "Drug/chemical-induced DM without complications",
    "E09.65": "Drug/chemical-induced DM with hyperglycemia",
    "E13.9": "Other specified DM without complications",
    # --- GESTATIONAL ---
    "O24.410": "Gestational DM in pregnancy, diet controlled",
    "O24.414": "Gestational DM in pregnancy, insulin controlled",
    "O24.419": "Gestational DM in pregnancy, unspecified control",
    # --- SUPPORTING CODES ---
    "Z79.4": "Long term (current) use of insulin",
    "Z79.84": "Long term (current) use of oral hypoglycemic drugs",
    "Z86.39": "Personal history of other endocrine, nutritional and metabolic disease",
    "R73.01": "Impaired fasting glucose",
    "R73.02": "Impaired glucose tolerance",
    "R80.8": "Other proteinuria",
    "R80.9": "Proteinuria, unspecified",
    "M86.9": "Osteomyelitis, unspecified",
}

print(f"Loaded {len(ICD10_REFERENCE)} ICD-10 reference codes")

Loaded 96 ICD-10 reference codes


## 4. Approach A: Rule-Based Extraction (Baseline)

Regex patterns + NegEx assertion detection + dictionary lookup.  
Serves as the measurable floor for LLM improvement.

In [54]:
# ============================================================
# NegEx: Simple assertion detection
# ============================================================
class NegEx:
    NEGATION = ["no ", "no evidence of", "denies", "denied", "without",
                "not ", "negative for", "rule out", "r/o", "never had",
                "no signs of", "absence of", "free of"]
    HISTORICAL = ["history of", "h/o", "hx of", "previous", "previously",
                  "resolved", "in the past", "past medical history", "pmh:"]
    FAMILY = ["family history", "fh:", "family hx", "mother had",
              "father had", "familial"]

    @staticmethod
    def check(text: str, entity_start: int, window: int = 60) -> str:
        pre = text[max(0, entity_start - window):entity_start].lower()
        for t in NegEx.FAMILY:
            if t in pre: return "family_history"
        for t in NegEx.HISTORICAL:
            if t in pre: return "historical"
        for t in NegEx.NEGATION:
            if t in pre: return "negated"
        return "affirmed"


# ============================================================
# Rule-Based Extractor
# ============================================================
class RuleBasedExtractor:
    def __init__(self):
        self.dm_patterns = [
            (r"diabetes\s+mellitus[\s,]*type\s*1", "type1"),
            (r"type\s*1\s+diabet\w*", "type1"),
            (r"T1DM|IDDM", "type1"),
            (r"diabetes\s+mellitus[\s,]*type\s*2", "type2"),
            (r"type\s*2\s+diabet\w*", "type2"),
            (r"T2DM|NIDDM", "type2"),
            (r"(?:diabetic\s+control|diabet(?:es|ic))", "unspecified"),
        ]
        self.comp_patterns = {
            "nephropathy": [r"diabetic\s+nephropath\w*", r"(?:kidney|renal)\s+(?:damage|disease)"],
            "neuropathy": [r"diabetic\s+neuropath\w*", r"neuropath\w*", r"(?:lower\s+extremity\s+)?numbness"],
            "retinopathy": [r"diabetic\s+retinopath\w*", r"retinopath\w*"],
            "hyperglycemia": [r"poorly\s+controlled", r"hyperglycemi\w*", r"(?:elevated|high)\s+(?:blood\s+)?(?:sugar|glucose)"],
            "hypoglycemia": [r"hypoglycemi\w*", r"low\s+blood\s+(?:sugar|glucose)\w*"],
            "ketoacidosis": [r"(?:diabetic\s+)?ketoacidosis", r"\bDKA\b"],
            "microalbuminuria": [r"(?:positive\s+)?microalbuminuri\w*"],
            "gastroparesis": [r"gastroparesis"],
            "osteomyelitis": [r"osteomyelitis"],
        }
        self.insulin_patterns = [
            r"(?:NPH|Lantus|Humalog|Novolog|Regular)\s*insulin",
            r"insulin\s+(?:regimen|glargine|lispro|aspart)",
            r"sliding\s+scale\s+(?:\w+\s+)?insulin",
        ]

        self.oral_med_patterns = [
            # Biguanides
            r"\bmetformin\b", r"\bglucophage\b", r"\bfortamet\b", r"\bglumetz a\b", r"\briomet\b",
            # Sulfonylureas
            r"\bglipizide\b", r"\bglucotrol\b",
            r"\bglyburide\b", r"\bglynase\b", r"\bmicronase\b", r"\bdiabeta\b",
            r"\bglimepiride\b", r"\bamaryl\b",
            # Thiazolidinediones
            r"\bpioglitazone\b", r"\bactos\b",
            r"\brosiglitazone\b", r"\bavandia\b",
            # DPP-4 inhibitors
            r"\bsitagliptin\b", r"\bjanuvia\b",
            r"\bsaxagliptin\b", r"\bonglyza\b",
            r"\blinagliptin\b", r"\btradjenta\b",
            r"\balogliptin\b", r"\bnesina\b",
            # SGLT2 inhibitors
            r"\bempagliflozin\b", r"\bjardiance\b",
            r"\bdapagliflozin\b", r"\bfarxiga\b",
            r"\bcanagliflozin\b", r"\binvokana\b",
            r"\bertugliflozin\b", r"\bsteglatro\b",
            # GLP-1 receptor agonists
            r"\bliraglutide\b", r"\bvictoza\b",
            r"\bsemaglutide\b", r"\bozempic\b", r"\brybelsus\b", r"\bwegovy\b",
            r"\bdulaglutide\b", r"\btrulicity\b",
            r"\bexenatide\b", r"\bbyetta\b", r"\bbydureon\b",
            r"\btirzepatide\b", r"\bmounjaro\b", r"\bzepbound\b",
            # Alpha-glucosidase inhibitors
            r"\bacarbose\b", r"\bprecose\b",
            r"\bmiglitol\b", r"\bglyset\b",
            # Meglitinides
            r"\brepaglinide\b", r"\bprandin\b",
            r"\bnateglinide\b", r"\bstarlix\b",
            # Combo pills (brand names)
            r"\bjanumet\b", r"\binvokamet\b", r"\bsynjardy\b",
            r"\bxigduo\b", r"\bglyxambi\b", r"\bjentadueto\b",
            # Generic phrase
            r"oral\s+hypoglycemic",
        ]

    def extract(self, text: str) -> list[dict]:
        dm_type = self._get_dm_type(text)
        if not dm_type:
            return []
        comps = self._get_complications(text)
        return self._map_codes(dm_type, comps, text)

    def _get_dm_type(self, text):
        for pat, dtype in self.dm_patterns:
            m = re.search(pat, text, re.IGNORECASE)
            if m and NegEx.check(text, m.start()) == "affirmed":
                return dtype if dtype != "unspecified" else "type2"
        return None

    def _get_complications(self, text):
        found = {}
        for name, patterns in self.comp_patterns.items():
            for pat in patterns:
                m = re.search(pat, text, re.IGNORECASE)
                if m:
                    assertion = NegEx.check(text, m.start())
                    if name not in found or assertion == "affirmed":
                        found[name] = {"assertion": assertion, "evidence": m.group(0)}
        return found

    def _has_insulin(self, text):
        return any(re.search(p, text, re.IGNORECASE) for p in self.insulin_patterns)
    
    def _has_oral_meds(self, text):
        return any(re.search(p, text, re.IGNORECASE) for p in self.oral_med_patterns)

    def _map_codes(self, dm_type, comps, text):
        codes = []
        pfx = "E10" if dm_type == "type1" else "E11"
        has_comp = False

        mapping = {
            "hyperglycemia": f"{pfx}.65", "hypoglycemia": f"{pfx}.649",
            "nephropathy": f"{pfx}.21", "neuropathy": f"{pfx}.40",
            "retinopathy": f"{pfx}.319", "ketoacidosis": f"{pfx}.10",
            "microalbuminuria": f"{pfx}.29", "osteomyelitis": f"{pfx}.69",
        }

        # Check HbA1c for hyperglycemia
        hba1c = re.search(r"A1[cC]\s*(?:of\s+|drawn\s+.*?is\s+|:?\s*)(\d+\.?\d*)", text, re.IGNORECASE)
        if hba1c:
            try:
                val = float(hba1c.group(1))
                if val > 9.0 and "hyperglycemia" not in comps:
                    comps["hyperglycemia"] = {"assertion": "affirmed", "evidence": f"HbA1c {val}"}
            except ValueError:
                pass

        for comp, info in comps.items():
            if info["assertion"] == "affirmed" and comp in mapping:
                c = mapping[comp]
                if c in ICD10_REFERENCE:
                    codes.append({"code": c, "display": ICD10_REFERENCE[c],
                                  "evidence": [info["evidence"]], "confidence": "high", "approach": "rule_based"})
                    has_comp = True

        # Standalone secondary codes
        if "microalbuminuria" in comps and comps["microalbuminuria"]["assertion"] == "affirmed":
            codes.append({"code": "R80.8", "display": ICD10_REFERENCE["R80.8"],
                          "evidence": [comps["microalbuminuria"]["evidence"]], "confidence": "medium", "approach": "rule_based"})

        if "osteomyelitis" in comps and comps["osteomyelitis"]["assertion"] == "affirmed":
            codes.append({"code": "M86.9", "display": ICD10_REFERENCE["M86.9"],
                          "evidence": [comps["osteomyelitis"]["evidence"]], "confidence": "medium", "approach": "rule_based"})

        if "gastroparesis" in comps and comps["gastroparesis"]["assertion"] in ("affirmed", "historical"):
            codes.append({"code": "Z86.39", "display": ICD10_REFERENCE["Z86.39"],
                          "evidence": [comps["gastroparesis"]["evidence"]], "confidence": "medium", "approach": "rule_based"})

        if self._has_insulin(text):
            codes.append({"code": "Z79.4", "display": ICD10_REFERENCE["Z79.4"],
                          "evidence": ["insulin use detected"], "confidence": "high", "approach": "rule_based"})
        
        if self._has_oral_meds(text):
            codes.append({"code": "Z79.84", "display": ICD10_REFERENCE.get("Z79.84", ""),
                          "evidence": ["oral hypoglycemic use detected"], "confidence": "high", "approach": "rule_based"})

        if not has_comp:
            codes.insert(0, {"code": f"{pfx}.9", "display": ICD10_REFERENCE.get(f"{pfx}.9", ""),
                             "evidence": ["no complications identified"], "confidence": "low", "approach": "rule_based"})

        return codes


extractor_a = RuleBasedExtractor()
print("=== APPROACH A: Rule-Based Results ===\n")
for test in TEST_SET:
    nid = test["note"]["id"]
    results = extractor_a.extract(test["note"]["text"])
    print(f"--- {nid} ---")
    for r in results:
        print(f"  {r['code']}: {r['display']}")
        print(f"    Evidence: {r['evidence']}")
    print()

=== APPROACH A: Rule-Based Results ===

--- note_13_Amazon_108411 ---
  E11.65: Type 2 DM with hyperglycemia
    Evidence: ['hyperglycemia']
  E11.649: Type 2 DM with hypoglycemia without coma
    Evidence: ['hypoglycemia']
  Z79.84: Long term (current) use of oral hypoglycemic drugs
    Evidence: ['oral hypoglycemic use detected']

--- note_01 ---
  E10.21: Type 1 DM with diabetic nephropathy
    Evidence: ['kidney damage']
  E10.40: Type 1 DM with diabetic neuropathy, unspecified
    Evidence: ['neuropathy']
  E10.65: Type 1 DM with hyperglycemia
    Evidence: ['poorly controlled']
  E10.649: Type 1 DM with hypoglycemia without coma
    Evidence: ['low blood sugar']
  E10.29: Type 1 DM with other diabetic kidney complication
    Evidence: ['positive microalbuminuria']
  R80.8: Other proteinuria
    Evidence: ['positive microalbuminuria']
  Z86.39: Personal history of other endocrine, nutritional and metabolic disease
    Evidence: ['gastroparesis']
  Z79.4: Long term (current) use of

## 5. Approach C: LLM Structured Extraction

Uses Claude API with a carefully crafted system prompt that includes:
- The full ICD-10 reference as a constraint
- Clinical coding rules (type inference, assertion, complication linking, specificity)
- Structured JSON output schema with evidence and reasoning

In [6]:
# ============================================================
# Build the LLM system prompt
# ============================================================

def build_system_prompt(ref: dict) -> str:
    ref_lines = [f"  {c}: {d}" for c, d in sorted(ref.items())]
    ref_block = chr(10).join(ref_lines)

    return f"""You are an expert medical coder specializing in ICD-10-CM coding from clinical notes.

TASK: Extract ALL relevant ICD-10 codes from the provided clinical note.

CODING RULES:

1. DIABETES TYPE INFERENCE:
   - If "type 1" or "type 2" is stated anywhere in the note, use that type
   - If only "diabetes" without type: adult (>=18) -> Type 2 (E11.x)
   - Look at insulin regimen, age, and clinical context for clues

2. ASSERTION DETECTION — for each finding determine:
   - "affirmed": Actively present / being treated this encounter
   - "negated": Explicitly denied ("no retinopathy", "denies")
   - "historical": Past/resolved ("history of", "previously had")
   - "family_history": In family member, not patient
   - "conditional": Under investigation ("rule out", "possible")
   ONLY code "affirmed" as active diagnoses. Historical may get Z-codes.

3. COMPLICATION LINKING:
   - Use combination codes (E10.40 for T1DM + neuropathy)
   - Do NOT code diabetes and complication separately
   - Coexisting but not direct DM complication -> use .69 "other specified"

4. SPECIFICITY — always use the MOST SPECIFIC code the note supports:
   - "moderate nonproliferative retinopathy with macular edema" -> E11.331
   - "diabetic polyneuropathy" -> E10.42 (not E10.40)
   - Only use unspecified when the note lacks detail

5. SUPPORTING CODES:
   - Z79.4: Long-term insulin use (when insulin meds are documented)
   - Z86.39: Personal history of metabolic disease (historical DM complications)
   - R80.8: Other proteinuria (standalone microalbuminuria)
   - M86.x: Osteomyelitis (when present)

6. HYPERGLYCEMIA vs HYPOGLYCEMIA:
   - "Poorly controlled" + high HbA1c/blood sugar -> hyperglycemia (.65)
   - "Low blood sugar/glucose", "glucoses in the 70s" -> hypoglycemia (.649/.641)
   - Read the CONTEXT — "poorly controlled" alone does not determine direction

VALID ICD-10 CODES — only return codes from this list:
{ref_block}

OUTPUT: Return ONLY a JSON array. Each element:
{{
  "code": "E11.65",
  "display": "Type 2 DM with hyperglycemia",
  "assertion": "affirmed",
  "evidence": ["exact quote from note supporting this code"],
  "confidence": "high|medium|low",
  "reasoning": "Brief explanation"
}}

CRITICAL: Return ONLY valid JSON array. No markdown fences, no explanation outside JSON."""


SYSTEM_PROMPT = build_system_prompt(ICD10_REFERENCE)
print(f"System prompt: {len(SYSTEM_PROMPT)} chars, includes {len(ICD10_REFERENCE)} reference codes")

System prompt: 7873 chars, includes 96 reference codes


In [ ]:
# ============================================================
# LLM Extraction via Claude API
# ============================================================

import subprocess

def extract_with_llm(note_text: str, system_prompt: str, api_key: str = None) -> list[dict]:
    """Extract ICD-10 codes using Claude API.

    Args:
        note_text: The clinical note text
        system_prompt: System prompt with coding rules and ICD-10 reference
        api_key: Anthropic API key. If None, reads from ANTHROPIC_API_KEY env var.

    Returns:
        List of extracted code dicts
    """
    key = api_key or os.environ.get("ANTHROPIC_API_KEY", "")
    if not key:
        print("ERROR: No API key. Set ANTHROPIC_API_KEY or pass api_key parameter.")
        return []

    payload = json.dumps({
        "model": "claude-sonnet-4-20250514",
        "max_tokens": 2000,
        "temperature": 0,
        "system": system_prompt,
        "messages": [{"role": "user",
                       "content": f"Extract all relevant ICD-10 codes from this clinical note:\n\n{note_text}"}]
    })

    result = subprocess.run(
        ["curl", "-s", "-X", "POST", "https://api.anthropic.com/v1/messages",
         "-H", "Content-Type: application/json",
         "-H", f"x-api-key: {key}",
         "-H", "anthropic-version: 2023-06-01",
         "-d", payload],
        capture_output=True, text=True, timeout=120
    )

    try:
        resp = json.loads(result.stdout)

        # Check for API errors
        if "error" in resp:
            print(f"API Error: {resp['error']}")
            return []

        text_out = "".join(b["text"] for b in resp.get("content", []) if b.get("type") == "text")
        text_out = text_out.strip()

        # Strip markdown fences if present
        if text_out.startswith("```"):
            text_out = text_out.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        codes = json.loads(text_out)
        for c in codes:
            c["approach"] = "llm"
        return codes
    except Exception as e:
        print(f"Parse error: {e}")
        print(f"Raw (first 500 chars): {result.stdout[:500]}")
        return []


print("LLM extraction function ready.")
print(f"ANTHROPIC_API_KEY set: {'Yes' if os.environ.get('ANTHROPIC_API_KEY') else 'No — set it before running'}")

## 6. Run LLM Extraction

Set your API key, then run the extraction.

In [ ]:
# ============================================================
# SET YOUR API KEY (uncomment one option)
# ============================================================

# Option 1: Set directly
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# Option 2: Already set in environment (nothing to do)

# ============================================================
# RUN LLM EXTRACTION
# ============================================================

llm_results = {}

if os.environ.get("ANTHROPIC_API_KEY"):
    for test in TEST_SET:
        nid = test["note"]["id"]
        print(f"\n--- Extracting: {nid} ---")
        codes = extract_with_llm(test["note"]["text"], SYSTEM_PROMPT)
        llm_results[nid] = codes
        for c in codes:
            print(f"  {c['code']}: {c['display']}")
            print(f"    Assertion: {c.get('assertion', 'N/A')}")
            print(f"    Evidence: {c.get('evidence', [])}")
            print(f"    Reasoning: {c.get('reasoning', '')}")
else:
    print("Skipped: Set ANTHROPIC_API_KEY first.")
    print("Run: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'")

## 7. Evaluation Framework

In [55]:
# ============================================================
# EVALUATION: Precision, Recall, F1 at code level
# ============================================================

def evaluate(predicted: list[dict], gold: list[dict], note_id: str) -> dict:
    pred_set = set(c["code"] for c in predicted)
    gold_set = set(c["code"] for c in gold)
    tp = pred_set & gold_set
    fp = pred_set - gold_set
    fn = gold_set - pred_set
    p = len(tp) / len(pred_set) if pred_set else 0
    r = len(tp) / len(gold_set) if gold_set else 0
    f1 = 2*p*r / (p+r) if (p+r) > 0 else 0
    return {"note_id": note_id, "tp": sorted(tp), "fp": sorted(fp), "fn": sorted(fn),
            "precision": round(p, 3), "recall": round(r, 3), "f1": round(f1, 3),
            "n_pred": len(pred_set), "n_gold": len(gold_set)}


def print_eval(e: dict, approach: str):
    print(f"\n{'='*55}")
    print(f"  {approach} | {e['note_id']}")
    print(f"{'='*55}")
    print(f"  Precision: {e['precision']:.1%}  Recall: {e['recall']:.1%}  F1: {e['f1']:.1%}")
    print(f"  Predicted: {e['n_pred']}  Gold: {e['n_gold']}")
    print(f"  TP: {e['tp']}")
    print(f"  FP: {e['fp']}")
    print(f"  FN: {e['fn']}")


print("Evaluation functions loaded.")

Evaluation functions loaded.


## 8. Rule-Based Evaluation (No API Key Needed)

In [56]:
# ============================================================
# EVALUATE APPROACH A
# ============================================================

rule_results = {}
for test in TEST_SET:
    nid = test["note"]["id"]
    results = extractor_a.extract(test["note"]["text"])
    rule_results[nid] = results
    e = evaluate(results, test["gold"]["codes"], nid)
    print_eval(e, "APPROACH A: Rule-Based")


  APPROACH A: Rule-Based | note_13_Amazon_108411
  Precision: 100.0%  Recall: 100.0%  F1: 100.0%
  Predicted: 3  Gold: 3
  TP: ['E11.649', 'E11.65', 'Z79.84']
  FP: []
  FN: []

  APPROACH A: Rule-Based | note_01
  Precision: 62.5%  Recall: 100.0%  F1: 76.9%
  Predicted: 8  Gold: 5
  TP: ['E10.29', 'E10.40', 'E10.65', 'R80.8', 'Z86.39']
  FP: ['E10.21', 'E10.649', 'Z79.4']
  FN: []

  APPROACH A: Rule-Based | note_02
  Precision: 100.0%  Recall: 100.0%  F1: 100.0%
  Predicted: 3  Gold: 3
  TP: ['E11.65', 'E11.69', 'M86.9']
  FP: []
  FN: []

  APPROACH A: Rule-Based | note_04
  Precision: 100.0%  Recall: 100.0%  F1: 100.0%
  Predicted: 2  Gold: 2
  TP: ['E11.649', 'Z79.4']
  FP: []
  FN: []


## 9. Side-by-Side Comparison

In [61]:
# ============================================================
# COMPARE APPROACHES — run after LLM extraction completes
# ============================================================

def compare(rule_res, llm_res, test_set):
    print(f"\n{'#'*55}")
    print(f"  SIDE-BY-SIDE COMPARISON")
    print(f"{'#'*55}")

    totals = {"rule": {"tp": 0, "fp": 0, "fn": 0}, "llm": {"tp": 0, "fp": 0, "fn": 0}}

    for test in test_set:
        nid = test["note"]["id"]
        gold = test["gold"]["codes"]
        re_ = evaluate(rule_res.get(nid, []), gold, nid)
        le_ = evaluate(llm_res.get(nid, []), gold, nid)

        totals["rule"]["tp"] += len(re_["tp"])
        totals["rule"]["fp"] += len(re_["fp"])
        totals["rule"]["fn"] += len(re_["fn"])
        totals["llm"]["tp"] += len(le_["tp"])
        totals["llm"]["fp"] += len(le_["fp"])
        totals["llm"]["fn"] += len(le_["fn"])

        print(f"\n--- {nid} ---")
        print(f"  Gold: {[c['code'] for c in gold]}")
        print(f"  {'Metric':<16} {'Rule':>10} {'LLM':>10}")
        print(f"  {'~'*36}")
        print(f"  {'Precision':<16} {re_['precision']:>9.1%} {le_['precision']:>9.1%}")
        print(f"  {'Recall':<16} {re_['recall']:>9.1%} {le_['recall']:>9.1%}")
        print(f"  {'F1':<16} {re_['f1']:>9.1%} {le_['f1']:>9.1%}")
        print(f"  {'TP':<16} {str(re_['tp']):>10} {str(le_['tp']):>10}")
        print(f"  {'FP':<16} {str(re_['fp']):>10} {str(le_['fp']):>10}")
        print(f"  {'FN':<16} {str(re_['fn']):>10} {str(le_['fn']):>10}")

    # Aggregate
    for approach in ["rule", "llm"]:
        t = totals[approach]
        p = t["tp"]/(t["tp"]+t["fp"]) if (t["tp"]+t["fp"]) else 0
        r = t["tp"]/(t["tp"]+t["fn"]) if (t["tp"]+t["fn"]) else 0
        f = 2*p*r/(p+r) if (p+r) else 0
        t["precision"], t["recall"], t["f1"] = p, r, f

    print(f"\n{'='*55}")
    print(f"  AGGREGATE (across {len(test_set)} notes)")
    print(f"{'='*55}")
    print(f"  {'Metric':<16} {'Rule':>10} {'LLM':>10}")
    print(f"  {'~'*36}")
    print(f"  {'Precision':<16} {totals['rule']['precision']:>9.1%} {totals['llm']['precision']:>9.1%}")
    print(f"  {'Recall':<16} {totals['rule']['recall']:>9.1%} {totals['llm']['recall']:>9.1%}")
    print(f"  {'F1':<16} {totals['rule']['f1']:>9.1%} {totals['llm']['f1']:>9.1%}")

llm_results = None
if llm_results:
    compare(rule_results, llm_results, TEST_SET)
else:
    print("LLM results not available yet. Run Section 6 first.")
    print("\nShowing Rule-Based results only:")
    for test in TEST_SET:
        nid = test["note"]["id"]
        e = evaluate(rule_results.get(nid, []), test["gold"]["codes"], nid)
        print(f"  {nid}: P={e['precision']:.1%} R={e['recall']:.1%} F1={e['f1']:.1%} TP={len(e['tp'])} FP={len(e['fp'])} FN={len(e['fn'])}")

LLM results not available yet. Run Section 6 first.

Showing Rule-Based results only:
  note_13_Amazon_108411: P=100.0% R=100.0% F1=100.0% TP=3 FP=0 FN=0
  note_01: P=62.5% R=100.0% F1=76.9% TP=5 FP=3 FN=0
  note_02: P=100.0% R=100.0% F1=100.0% TP=3 FP=0 FN=0
  note_04: P=100.0% R=100.0% F1=100.0% TP=2 FP=0 FN=0


## 10. Validation Layer (LLM Failure Mitigation)

Post-processing to catch hallucinated codes (F1), over-inference (F2),
assertion errors (F3), and specificity collapse (F4).

In [ ]:
# ============================================================
# VALIDATION LAYER
# ============================================================

class CodeValidator:
    def __init__(self, ref: dict):
        self.ref = ref

    def validate(self, codes: list[dict], note_text: str) -> list[dict]:
        clean = []
        for c in codes:
            issues = self._check(c, note_text)
            if issues["valid"]:
                clean.append(c)
            else:
                print(f"  REJECTED {c['code']}: {issues['reason']}")
                if issues.get("suggestion"):
                    print(f"    -> Suggestion: {issues['suggestion']}")
        return clean

    def _check(self, c, text):
        code = c.get("code", "")
        r = {"code": code, "valid": True, "reason": "", "suggestion": ""}

        # F1: Code existence
        if code not in self.ref:
            r["valid"] = False
            r["reason"] = f"Not in reference (hallucination?)"
            nearest = self._nearest(code)
            if nearest:
                r["suggestion"] = f"Nearest valid: {nearest} ({self.ref[nearest]})"
            return r

        # F2: Evidence in note
        evidence = c.get("evidence", [])
        if evidence and not any(e.lower() in text.lower() for e in evidence):
            r["reason"] = "Evidence not found verbatim (possible over-inference)"

        # F3: Assertion cross-check
        if c.get("assertion") == "affirmed" and evidence:
            for e in evidence:
                pos = text.lower().find(e.lower())
                if pos >= 0:
                    neg_check = NegEx.check(text, pos)
                    if neg_check != "affirmed":
                        r["reason"] = f"LLM=affirmed but NegEx={neg_check} for '{e}'"

        # F4: Specificity warning
        if code.endswith(".9") or (len(code) > 4 and code[-1] == "9" and code[-2] == "."):
            r["reason"] = "Unspecified code — check for more specific option"

        return r

    def _nearest(self, code):
        pfx = code
        while len(pfx) > 3:
            pfx = pfx[:-1]
            for v in self.ref:
                if v.startswith(pfx):
                    return v
        return None


validator = CodeValidator(ICD10_REFERENCE)
print("Validator ready.\\n")

# Demo
print("--- Validating Rule-Based results ---")
for test in TEST_SET:
    nid = test["note"]["id"]
    print(f"\\n{nid}:")
    validated = validator.validate(rule_results.get(nid, []), test["note"]["text"])
    print(f"  {len(validated)} / {len(rule_results.get(nid, []))} codes passed validation")

## 11. Note Loader Utility

In [ ]:
# ============================================================
# Load new notes from your annotation JSON format
# ============================================================

def normalize_code(raw: str) -> str:
    """E1065 -> E10.65, E11649 -> E11.649, R808 -> R80.8"""
    if "." in raw:
        return raw
    if len(raw) >= 4:
        return f"{raw[:3]}.{raw[3:]}"
    return raw


def load_note(note_path: str, ann_path: str) -> dict:
    """Load note + annotation files and return test entry."""
    with open(note_path) as f:
        note = json.load(f)
    with open(ann_path) as f:
        ann = json.load(f)

    if not ann.get("completed", False):
        print(f"WARNING: {ann.get('doc_id')} not marked completed")

    span_map = {s["id"]: s for s in ann.get("spans", [])}

    gold_codes = []
    for da in ann.get("document_annotations", []):
        concept = da.get("concept", {})
        code = normalize_code(concept.get("code", ""))
        evidence = [span_map[sid]["text"] for sid in da.get("evidence_span_ids", []) if sid in span_map]
        gold_codes.append({
            "code": code, "display": concept.get("display", ""),
            "evidence": evidence, "reasoning": da.get("note", "")
        })

    return {
        "note": note,
        "gold": {"note_id": note.get("id", ""), "codes": gold_codes}
    }


# Usage:
# entry = load_note("/path/to/note_5.json", "/path/to/note_5_ann.json")
# TEST_SET.append(entry)

print("Note loader ready.")
print("\nCode normalization examples:")
for raw in ["E1065", "E11649", "R808", "Z8639", "M869", "E1029", "E1040", "Z794"]:
    print(f"  {raw} -> {normalize_code(raw)}")

## 12. Summary & Next Steps

### What this notebook provides:
- **2 gold-standard notes** with human annotations (extensible via loader)
- **Approach A (Rule-based)** — deterministic baseline with regex + NegEx
- **Approach C (LLM)** — Claude API structured extraction with clinical coding rules
- **Evaluation** — P/R/F1 per note + aggregate comparison
- **Validation layer** — catches hallucinated codes, over-inference, assertion errors
- **Note loader** — plug in more annotated notes from your JSON format

### To run:
1. Cells 1-4, 7-8: No API key needed (rule-based baseline)
2. Cell 6: Add API key, run LLM extraction
3. Cell 9: Side-by-side comparison
4. Cell 10: Validation on LLM results

### Extending:
- More notes: Use `load_note()` in Cell 11
- Qwen3 1.7B: Replace `extract_with_llm` curl call with llama-cpp-python
- Majority voting: Call `extract_with_llm` 3x, take codes appearing in >= 2 runs
- Hybrid: Route simple notes -> rules, complex -> LLM

# Gold notes Conversion - Util

In [16]:
#!/usr/bin/env python3
"""Convert note_*_ann.json to gold standard format for ICD-10 evaluation."""

import json
import sys
import os


def normalize_code(raw: str) -> str:
    """E1065 -> E10.65, E11649 -> E11.649, R808 -> R80.8, Z794 -> Z79.4"""
    if "." in raw:
        return raw
    if len(raw) >= 4:
        return f"{raw[:3]}.{raw[3:]}"
    return raw


def convert(ann_path: str, output_path: str = None) -> dict:
    """Convert annotation JSON to gold standard format.

    Args:
        ann_path: Path to note_*_ann.json file
        output_path: Optional output path. If None, derives from input
                     (note_1_ann.json -> note_1_gold.json)
    Returns:
        The gold standard dict
    """
    with open(ann_path) as f:
        ann = json.load(f)

    if not ann.get("completed", False):
        print(f"WARNING: {ann.get('doc_id', 'unknown')} is not marked as completed")

    # Build span lookup
    span_map = {s["id"]: s for s in ann.get("spans", [])}

    # Convert each document annotation to a gold code entry
    codes = []
    for da in ann.get("document_annotations", []):
        concept = da.get("concept", {})
        code = normalize_code(concept.get("code", ""))

        # Collect unique evidence texts from spans (preserve order, deduplicate)
        evidence = []
        seen = set()
        for sid in da.get("evidence_span_ids", []):
            if sid in span_map:
                text = span_map[sid]["text"]
                if text not in seen:
                    evidence.append(text)
                    seen.add(text)

        codes.append({
            "code": code,
            "display": concept.get("display", ""),
            "evidence": evidence,
            "reasoning": da.get("note", "")
        })

    gold = {
        "note_id": ann.get("doc_id", ""),
        "codes": codes
    }

    # Derive output path if not provided
    if not output_path:
        base = ann_path.replace("_ann.json", "_gold.json")
        output_path = base

    with open(output_path, "w") as f:
        json.dump(gold, f, indent=2)

    print(f"Converted: {ann_path} -> {output_path}")
    print(f"  Note ID: {gold['note_id']}")
    print(f"  Codes: {len(codes)}")
    for c in codes:
        print(f"    {c['code']}: {c['display']}")

    return gold


# if __name__ == "__main__":
#     if len(sys.argv) < 2:
#         print("Usage: python convert_ann_to_gold.py <ann.json> [output.json]")
#         print("Example: python convert_ann_to_gold.py note_1_ann.json note_1_gold.json")
#         sys.exit(1)

ann_file = sys.argv[1]
out_file = sys.argv[2] if len(sys.argv) > 2 else None
convert(r"D:\Projects\ClinicalNLP\test_docs\note_13_ann.json", r"D:\Projects\ClinicalNLP\test_docs\note_13_ann_out.json")

Converted: D:\Projects\ClinicalNLP\test_docs\note_13_ann.json -> D:\Projects\ClinicalNLP\test_docs\note_13_ann_out.json
  Note ID: note_13_Amazon_108411
  Codes: 3
    Z79.84: Long term (current) use of oral hypoglycemic drugs
    E11.65: Type 2 diabetes mellitus with hyperglycemia
    E11.649: Type 2 diabetes mellitus with hypoglycemia without coma


{'note_id': 'note_13_Amazon_108411',
 'codes': [{'code': 'Z79.84',
   'display': 'Long term (current) use of oral hypoglycemic drugs',
   'evidence': ['Farxiga',
    'glimepiride',
    'Glimepiride',
    'metformin',
    'Metformin'],
   'reasoning': ''},
  {'code': 'E11.65',
   'display': 'Type 2 diabetes mellitus with hyperglycemia',
   'evidence': ['diabetes',
    'diabetes mellitus',
    'Diabetes mellitus',
    'Type 2',
    'hyperglycemia'],
   'reasoning': ''},
  {'code': 'E11.649',
   'display': 'Type 2 diabetes mellitus with hypoglycemia without coma',
   'evidence': ['diabetes',
    'diabetes mellitus',
    'Diabetes mellitus',
    'Type 2',
    'hypoglycemia'],
   'reasoning': ''}]}